# Pruning — CNN + Channel-Attention + SVM

Loads the *exact same* model produced by **training.ipynb** from `saved_models_original/`, evaluates it (Pipeline A — Original),
then applies magnitude pruning (Pipeline B — Pruned) and reports all required metrics.


In [ ]:
import os, time, gzip, shutil, json
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Layer, Conv2D, MaxPooling2D, Flatten, Dense,
                                     GlobalAveragePooling2D, Reshape, Multiply, Input)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# ----------------------------------------------------------------------------
# Custom ChannelAttention layer (must be identical to training.ipynb so that
# the saved .keras model can be loaded back successfully).
# ----------------------------------------------------------------------------
class ChannelAttention(Layer):
    """Channel Attention Module — same as used in training."""
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels      = input_shape[-1]
        self.gap      = GlobalAveragePooling2D()
        self.dense1   = Dense(max(1, channels // self.ratio), activation="relu")
        self.dense2   = Dense(channels, activation="sigmoid")
        self.reshape  = Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        attn = self.gap(x)
        attn = self.dense1(attn)
        attn = self.dense2(attn)
        attn = self.reshape(attn)
        return Multiply()([x, attn])

    def get_config(self):
        config = super().get_config()
        config.update({"ratio": self.ratio})
        return config

CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention}


In [ ]:
import random, os
import numpy as np
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print(f"[INFO] enable_op_determinism: {e}")

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except RuntimeError: pass
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("Running on CPU.")


In [ ]:
# ============================================================================
# UNIFIED EVALUATION HELPERS  (same definitions across all 4 notebooks)
# ============================================================================

def get_file_size_kb(path):
    """File size in KB, or 0.0 if missing."""
    return os.path.getsize(path) / 1024.0 if os.path.exists(path) else 0.0

def fmt_size(kb):
    if kb < 1024: return f"{kb:.2f} KB"
    return f"{kb/1024:.2f} MB"

def evaluate_pipeline(extractor, svm_clf, scaler, X_test, y_test_int,
                      class_names, label="Model", warmup=True):
    """Runs feature extraction + scaling + SVM prediction.
    Returns a dict containing accuracy, sensitivity, specificity, F1, AUC,
    confusion matrix, feature-extraction time, inference time."""

    # warm-up so first-call graph compilation does not pollute timing
    if warmup:
        try:
            _ = extractor.predict(X_test[:2], verbose=0)
        except Exception:
            pass

    # ---- timing: feature extraction ----
    t0 = time.perf_counter()
    X_test_feat = extractor.predict(X_test, verbose=0)
    feat_time = time.perf_counter() - t0

    # ---- timing: scaling + SVM inference ----
    t0 = time.perf_counter()
    X_test_scaled = scaler.transform(X_test_feat)
    y_pred        = svm_clf.predict(X_test_scaled)
    y_pred_proba  = svm_clf.predict_proba(X_test_scaled)
    inf_time      = time.perf_counter() - t0

    n_classes = len(class_names)
    cm        = confusion_matrix(y_test_int, y_pred, labels=list(range(n_classes)))

    sens, spec, prec, f1s = [], [], [], []
    for i in range(n_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FN - FP
        s  = TP/(TP+FN) if (TP+FN)>0 else 0.0
        sp = TN/(TN+FP) if (TN+FP)>0 else 0.0
        p  = TP/(TP+FP) if (TP+FP)>0 else 0.0
        f  = 2*p*s/(p+s) if (p+s)>0 else 0.0
        sens.append(s); spec.append(sp); prec.append(p); f1s.append(f)

    y_test_bin   = label_binarize(y_test_int, classes=list(range(n_classes)))
    auc_per_cls  = roc_auc_score(y_test_bin, y_pred_proba, average=None , multi_class="ovr")
    auc_macro    = roc_auc_score(y_test_bin, y_pred_proba, average="macro", multi_class="ovr")
    auc_micro    = roc_auc_score(y_test_bin, y_pred_proba, average="micro", multi_class="ovr")
    accuracy     = accuracy_score(y_test_int, y_pred)

    return {
        "label"        : label,
        "accuracy"     : accuracy,
        "sensitivity"  : list(sens),
        "specificity"  : list(spec),
        "precision"    : list(prec),
        "f1"           : list(f1s),
        "auc_per_class": list(auc_per_cls),
        "auc_macro"    : auc_macro,
        "auc_micro"    : auc_micro,
        "cm"           : cm,
        "feat_time"    : feat_time,
        "inf_time"     : inf_time,
        "y_pred"       : y_pred,
        "y_pred_proba" : y_pred_proba,
    }


def per_class_metrics_df(metrics, class_names):
    rows = []
    rows.append(["Sensitivity"] + [f"{v:.4f}" for v in metrics["sensitivity" ]] + [f'{np.mean(metrics["sensitivity" ]):.4f}'])
    rows.append(["Specificity"] + [f"{v:.4f}" for v in metrics["specificity" ]] + [f'{np.mean(metrics["specificity" ]):.4f}'])
    rows.append(["Precision"  ] + [f"{v:.4f}" for v in metrics["precision"   ]] + [f'{np.mean(metrics["precision"   ]):.4f}'])
    rows.append(["F1-Score"   ] + [f"{v:.4f}" for v in metrics["f1"          ]] + [f'{np.mean(metrics["f1"          ]):.4f}'])
    rows.append(["AUC"        ] + [f"{v:.4f}" for v in metrics["auc_per_class"]] + [f'{np.mean(metrics["auc_per_class"]):.4f}'])
    return pd.DataFrame(rows, columns=["Metric"] + class_names + ["Mean"])


def print_evaluation_block(metrics, class_names, title="EVALUATION METRICS"):
    print("=" * 78)
    print(f"  {title} — {metrics['label']}")
    print("=" * 78)
    df = per_class_metrics_df(metrics, class_names)
    print(df.to_string(index=False))
    print("-" * 78)
    print(f"Accuracy        : {metrics['accuracy' ]:.4f}")
    print(f"AUC (macro avg) : {metrics['auc_macro']:.4f}")
    print(f"AUC (micro avg) : {metrics['auc_micro']:.4f}")
    print("\nConfusion Matrix (rows=true, cols=pred):")
    cm_df = pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)
    print(cm_df.to_string())
    print("=" * 78)


def resource_table(name_size_param_list):
    """Builds a resource-comparison table.
    Each entry: (label, size_kb, n_params, feat_time_s, inf_time_s)"""
    df = pd.DataFrame(
        name_size_param_list,
        columns=["Model", "Size (KB)", "Parameters",
                 "Feature Extraction (s)", "Inference (s)"]
    )
    return df


def plot_confusion_matrix(metrics, class_names, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(metrics["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title or f"Confusion Matrix — {metrics['label']}")
    return ax


In [ ]:
# ============================================================================
# LOAD DATASET (train / valid / test)
# ============================================================================
dataset_base_path = "./dataset_processed2"
img_size          = 224
categories        = ["Bengin cases", "Malignant cases", "Normal cases"]
class_names       = ["Bengin", "Malignant", "Normal"]
num_classes       = len(categories)

def load_split_data(split_path, categories):
    X, y = [], []
    for class_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_path, cat)
        if not os.path.isdir(cat_path):
            continue
        for fn in sorted(os.listdir(cat_path)):
            if not fn.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img = cv2.imread(os.path.join(cat_path, fn))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            img = img.astype(np.float32) / 255.0
            X.append(img); y.append(class_idx)
    return np.array(X, dtype=np.float32), np.array(y)

print("Loading dataset…")
X_train, y_train_labels = load_split_data(os.path.join(dataset_base_path, "train"), categories)
X_valid, y_valid_labels = load_split_data(os.path.join(dataset_base_path, "valid"), categories)
X_test , y_test_labels  = load_split_data(os.path.join(dataset_base_path, "test" ), categories)

y_train     = to_categorical(y_train_labels, num_classes=num_classes)
y_valid     = to_categorical(y_valid_labels, num_classes=num_classes)
y_test      = to_categorical(y_test_labels , num_classes=num_classes)
y_train_int = y_train_labels
y_valid_int = y_valid_labels
y_test_int  = y_test_labels

print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")


In [ ]:
# ============================================================================
# LOAD ORIGINAL ARTEFACTS  (saved by training.ipynb)
# ============================================================================
ORIG_FOLDER = "saved_models_original"

if not os.path.isdir(ORIG_FOLDER):
    raise FileNotFoundError(
        f"Folder '{ORIG_FOLDER}' tidak ditemukan. "
        "Jalankan training.ipynb lebih dulu agar model & artefak tersimpan."
    )

cnn_path       = os.path.join(ORIG_FOLDER, "cnn_attention_model.keras")
extractor_path = os.path.join(ORIG_FOLDER, "feature_extractor.keras")
svm_path       = os.path.join(ORIG_FOLDER, "svm_classifier.pkl")
scaler_path    = os.path.join(ORIG_FOLDER, "feature_scaler.pkl")

print(f"Loading artefacts from {ORIG_FOLDER}/ …")
model_orig     = tf.keras.models.load_model(cnn_path      , custom_objects=CUSTOM_OBJECTS)
extractor_orig = tf.keras.models.load_model(extractor_path, custom_objects=CUSTOM_OBJECTS)
svm_orig       = joblib.load(svm_path)
scaler_orig    = joblib.load(scaler_path)

orig_size_cnn       = get_file_size_kb(cnn_path)
orig_size_extractor = get_file_size_kb(extractor_path)
orig_size_svm       = get_file_size_kb(svm_path)
orig_size_scaler    = get_file_size_kb(scaler_path)

# Tabel ukuran artefak yang di-load
size_load_df = pd.DataFrame([
    ["cnn_attention_model.keras", orig_size_cnn      , get_file_size_kb(cnn_path      )],
    ["feature_extractor.keras"  , orig_size_extractor, get_file_size_kb(extractor_path)],
    ["svm_classifier.pkl"       , orig_size_svm      , get_file_size_kb(svm_path      )],
    ["feature_scaler.pkl"       , orig_size_scaler   , get_file_size_kb(scaler_path   )],
], columns=["File", "Size (KB)", "Bytes/1024"])
print("\nLoaded artefact sizes:")
print(size_load_df.to_string(index=False))
print(f"\nCNN params         : {model_orig.count_params():,}")
print(f"Extractor params   : {extractor_orig.count_params():,}")


In [ ]:
# ============================================================================
# PRUNING HELPERS
# ============================================================================
try:
    import tensorflow_model_optimization as tfmot
    _TFMOT_AVAILABLE = True
    print("tensorflow_model_optimization detected — using tfmot pruning API.")
except Exception as e:
    _TFMOT_AVAILABLE = False
    print(f"tfmot not available ({e}). Falling back to manual magnitude pruning.")


def _gzip_file(src, dst):
    with open(src, "rb") as fi, gzip.open(dst, "wb") as fo:
        shutil.copyfileobj(fi, fo)


def _build_pruning_masks(weight_list, target_sparsity=0.80):
    masks = []
    for w in weight_list:
        if w.ndim >= 2 and w.size > 0:
            thr = np.percentile(np.abs(w).flatten(), target_sparsity * 100.0)
            masks.append((np.abs(w) >= thr).astype(np.float32))
        else:
            masks.append(np.ones_like(w, dtype=np.float32))
    return masks


def _apply_masks(weight_list, masks):
    return [w * m for w, m in zip(weight_list, masks)]


class SparsityMaskCallback(tf.keras.callbacks.Callback):
    def __init__(self, masks): super().__init__(); self.masks = masks
    def on_train_batch_end(self, batch, logs=None):
        self.model.set_weights(_apply_masks(self.model.get_weights(), self.masks))


def _create_pruned_model(model, final_sparsity, prune_epochs, batch_size, X_train):
    if _TFMOT_AVAILABLE:
        try:
            end_step = int(np.ceil(len(X_train) / batch_size) * prune_epochs)
            sched = tfmot.sparsity.keras.PolynomialDecay(
                initial_sparsity=0.30, final_sparsity=final_sparsity,
                begin_step=0, end_step=max(1, end_step))
            pm = tfmot.sparsity.keras.prune_low_magnitude(model, pruning_schedule=sched)
            return pm, [tfmot.sparsity.keras.UpdatePruningStep()], "tfmot"
        except Exception as e:
            print(f"[Pruning] tfmot fallback because: {e}")

    # manual mask fallback
    pm = tf.keras.models.clone_model(model)
    pm.build(model.input_shape)
    pm.set_weights(model.get_weights())
    masks = _build_pruning_masks(pm.get_weights(), target_sparsity=final_sparsity)
    pm.set_weights(_apply_masks(pm.get_weights(), masks))
    return pm, [SparsityMaskCallback(masks)], "manual-mask"


def setup_pruning_pipeline(model, X_train, X_valid, y_train, y_valid,
                           batch_size=32, prune_epochs=5, final_sparsity=0.80):
    os.makedirs("artifacts", exist_ok=True)

    pruned_model, prune_cbs, mode = _create_pruned_model(
        model, final_sparsity, prune_epochs, batch_size, X_train)
    pruned_model.compile(optimizer="adam", loss="categorical_crossentropy",
                         metrics=["accuracy"])
    print(f"[Pruning] mode = {mode}")

    t0 = time.perf_counter()
    hist = pruned_model.fit(
        X_train, y_train, epochs=prune_epochs, batch_size=batch_size,
        validation_data=(X_valid, y_valid),
        callbacks=prune_cbs + [EarlyStopping(patience=2, monitor="val_loss",
                                             restore_best_weights=True)],
        verbose=1,
    )
    finetune_time = time.perf_counter() - t0

    if _TFMOT_AVAILABLE and mode == "tfmot":
        stripped = tfmot.sparsity.keras.strip_pruning(pruned_model)
    else:
        stripped = pruned_model
    stripped.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

    extractor_pruned = Model(inputs=stripped.input,
                             outputs=stripped.get_layer("feature_layer").output)

    cnn_pruned_path  = "artifacts/cnn_attention_pruned.keras"
    ext_pruned_path  = "artifacts/feature_extractor_pruned.keras"
    cnn_pruned_gz    = "artifacts/cnn_attention_pruned.keras.gz"
    ext_pruned_gz    = "artifacts/feature_extractor_pruned.keras.gz"
    stripped         .save(cnn_pruned_path)
    extractor_pruned .save(ext_pruned_path)
    _gzip_file(cnn_pruned_path, cnn_pruned_gz)
    _gzip_file(ext_pruned_path, ext_pruned_gz)

    paths = dict(cnn=cnn_pruned_path, extractor=ext_pruned_path,
                 cnn_gz=cnn_pruned_gz, extractor_gz=ext_pruned_gz)
    return paths, stripped, extractor_pruned, hist, finetune_time


In [ ]:
# ============================================================================
# PIPELINE A: ORIGINAL  (no retraining — uses loaded SVM + scaler)
# ============================================================================
metrics_orig = evaluate_pipeline(
    extractor_orig, svm_orig, scaler_orig,
    X_test, y_test_int, class_names, label="Original (loaded)"
)
print_evaluation_block(metrics_orig, class_names, "ORIGINAL — TEST SET")

# ============================================================================
# PIPELINE B: PRUNED MODEL  (CNN pruned, SVM retrained on new features)
# ============================================================================
print("\n" + "=" * 78)
print("APPLYING MAGNITUDE PRUNING TO CNN …")
print("=" * 78)

paths_pruned, model_pruned, extractor_pruned, hist_prune, prune_time = setup_pruning_pipeline(
    model_orig, X_train, X_valid, y_train, y_valid,
    batch_size=32, prune_epochs=5, final_sparsity=0.80,
)
print(f"[Pruning] fine-tune time : {prune_time:.3f} s")

# refit SVM + scaler on features extracted from the pruned CNN
X_tr_p = extractor_pruned.predict(X_train, verbose=0)
X_va_p = extractor_pruned.predict(X_valid, verbose=0)

scaler_pruned = StandardScaler().fit(np.vstack([X_tr_p, X_va_p]))
X_tr_ps = scaler_pruned.transform(X_tr_p)
X_va_ps = scaler_pruned.transform(X_va_p)

svm_pruned = SVC(kernel="rbf", C=1.0, gamma="scale",
                 probability=True, random_state=42)
svm_pruned.fit(np.vstack([X_tr_ps, X_va_ps]),
               np.concatenate([y_train_int, y_valid_int]))

joblib.dump(svm_pruned   , "artifacts/svm_pruned.pkl"   )
joblib.dump(scaler_pruned, "artifacts/scaler_pruned.pkl")

metrics_pruned = evaluate_pipeline(
    extractor_pruned, svm_pruned, scaler_pruned,
    X_test, y_test_int, class_names, label="Pruned (sparsity=0.80)"
)
print_evaluation_block(metrics_pruned, class_names, "PRUNED — TEST SET")


In [ ]:
# ============================================================================
# RESOURCE / SIZE / METRIC COMPARISON TABLES
# ============================================================================
size_orig_total       = orig_size_cnn + orig_size_svm + orig_size_scaler
size_pruned_kb        = get_file_size_kb(paths_pruned["cnn"])
size_pruned_gz_kb     = get_file_size_kb(paths_pruned["cnn_gz"])
size_pruned_total     = (size_pruned_kb +
                         get_file_size_kb("artifacts/svm_pruned.pkl") +
                         get_file_size_kb("artifacts/scaler_pruned.pkl"))
size_pruned_gz_total  = (size_pruned_gz_kb +
                         get_file_size_kb("artifacts/svm_pruned.pkl") +
                         get_file_size_kb("artifacts/scaler_pruned.pkl"))

# 1) Resource comparison
res_df = resource_table([
    ("Original",
     size_orig_total, model_orig.count_params(),
     metrics_orig["feat_time"], metrics_orig["inf_time"]),
    ("Pruned (.keras)",
     size_pruned_total, model_pruned.count_params(),
     metrics_pruned["feat_time"], metrics_pruned["inf_time"]),
    ("Pruned (.keras.gz)",
     size_pruned_gz_total, model_pruned.count_params(),
     metrics_pruned["feat_time"], metrics_pruned["inf_time"]),
])
print("\n" + "=" * 78)
print("RESOURCE COMPARISON")
print("=" * 78)
print(res_df.to_string(index=False))

# 2) Detailed file-size comparison
size_df = pd.DataFrame([
    ["CNN Attention (main)" , orig_size_cnn      , size_pruned_kb     , size_pruned_gz_kb],
    ["SVM Classifier"       , orig_size_svm      , get_file_size_kb("artifacts/svm_pruned.pkl"   ), get_file_size_kb("artifacts/svm_pruned.pkl"   )],
    ["Feature Scaler"       , orig_size_scaler   , get_file_size_kb("artifacts/scaler_pruned.pkl"), get_file_size_kb("artifacts/scaler_pruned.pkl")],
], columns=["Komponen", "Awal (KB)", "Pruned (KB)", "Pruned+GZIP (KB)"])
size_df.loc[len(size_df)] = ["TOTAL",
                             size_df["Awal (KB)"].sum(),
                             size_df["Pruned (KB)"].sum(),
                             size_df["Pruned+GZIP (KB)"].sum()]
print("\nFILE-LEVEL SIZE COMPARISON")
print(size_df.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

# 3) Side-by-side metric summary
summary_df = pd.DataFrame({
    "Metric"   : ["Accuracy", "Sensitivity (mean)", "Specificity (mean)",
                  "F1-Score (mean)", "AUC (macro)"],
    "Original" : [
        f'{metrics_orig["accuracy"]:.4f}',
        f'{np.mean(metrics_orig["sensitivity"]):.4f}',
        f'{np.mean(metrics_orig["specificity"]):.4f}',
        f'{np.mean(metrics_orig["f1"          ]):.4f}',
        f'{metrics_orig["auc_macro"]:.4f}',
    ],
    "Pruned"   : [
        f'{metrics_pruned["accuracy"]:.4f}',
        f'{np.mean(metrics_pruned["sensitivity"]):.4f}',
        f'{np.mean(metrics_pruned["specificity"]):.4f}',
        f'{np.mean(metrics_pruned["f1"          ]):.4f}',
        f'{metrics_pruned["auc_macro"]:.4f}',
    ],
})
print("\n" + "=" * 78)
print("METRIC SUMMARY (Original vs Pruned)")
print("=" * 78)
print(summary_df.to_string(index=False))

# 4) Confusion-matrix figure side-by-side
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_confusion_matrix(metrics_orig  , class_names, ax=axes[0])
plot_confusion_matrix(metrics_pruned, class_names, ax=axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================================
# SAVE PRUNED ARTEFACTS TO A NEW FOLDER
# ============================================================================
from datetime import datetime
out_folder = f"saved_models_pruning_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_folder, exist_ok=True)

shutil.copy2(paths_pruned["cnn"]      , os.path.join(out_folder, "cnn_attention_model_pruned.keras"))
shutil.copy2(paths_pruned["extractor"], os.path.join(out_folder, "feature_extractor_pruned.keras"))
shutil.copy2("artifacts/svm_pruned.pkl"   , os.path.join(out_folder, "svm_classifier_pruned.pkl"))
shutil.copy2("artifacts/scaler_pruned.pkl", os.path.join(out_folder, "feature_scaler_pruned.pkl"))

with open(os.path.join(out_folder, "training_history_pruned.json"), "w") as f:
    json.dump({k: [float(x) for x in v] for k, v in hist_prune.history.items()},
              f, indent=2)

print(f"Pruned artefacts saved to {out_folder}/")
for fn in sorted(os.listdir(out_folder)):
    fp = os.path.join(out_folder, fn)
    print(f"  {fn:<40} {get_file_size_kb(fp):>10.2f} KB")
